# 01 — Supervised Fine-Tuning

Fine-tunes `distilgpt2` on a synthetic Hinge bio dataset using LoRA + TRL's `SFTTrainer`.
This is the first stage of the RLHF pipeline: teaching the model the output format and tone
before reward-based alignment.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jonasneves/aipi590-challenge-2/blob/main/notebooks/01_sft.ipynb)

In [ ]:
# Setup — install dependencies, clone repo, configure environment
!pip install -q trl peft transformers datasets accelerate bitsandbytes

import os
from google.colab import userdata
os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")

import sys
sys.path.insert(0, "/content/aipi590-challenge-2")

from src.colab_utils import prepare_notebook, publish_artifacts

repo_root, paths = prepare_notebook("aipi590-challenge-2")
print(f"Repo root: {repo_root}")
print(f"Paths: {paths}")

In [ ]:
# Build synthetic SFT training dataset — 20 person/bio pairs
import json
from pathlib import Path

SYNTHETIC_BIOS = [
    {"person": "Software engineer, 28, NYC, climbs on weekends, makes sourdough", "bio": "Will debug your code and your sourdough starter. Currently training for my first climbing comp — send beta or send memes, both appreciated."},
    {"person": "Nurse, 26, Chicago, reads sci-fi, runs half marathons", "bio": "By day I keep humans alive. By night I find out if the androids dream of electric sheep. Currently training for Chicago Half — race-day snacks non-negotiable."},
    {"person": "High school history teacher, 31, Austin, plays guitar, homebrews beer", "bio": "I teach the Civil War by day and brew an IPA inspired by it by night. My students think I'm cool; my homebrew club is still deciding."},
    {"person": "UX designer, 27, SF, surfs, obsessed with typography", "bio": "I'll redesign your app before dinner and catch a wave after. Yes, I notice when a font is wrong. No, I can't turn it off."},
    {"person": "Grad student in ecology, 25, Seattle, birdwatcher, plays chess", "bio": "I can name the bird outside your window. I will also beat you at chess and feel only slightly bad about it."},
    {"person": "Pastry chef, 29, New Orleans, jazz fan, does pottery", "bio": "I make croissants that take three days and pots that take three months. Both are worth the wait. Jazz optional but preferred."},
    {"person": "Aerospace engineer, 30, LA, amateur astronomer, plays volleyball", "bio": "I spend my days making things go to space and my nights watching them once they get there. Weekend volleyball keeps me grounded — literally."},
    {"person": "Documentary filmmaker, 27, Portland, vegan, forages for mushrooms", "bio": "I will film things that matter and forage the mushrooms for dinner. Yes, they're safe. Mostly."},
    {"person": "Financial analyst, 32, Boston, into CrossFit, reads philosophy", "bio": "I model risk for a living, which means I'm also very aware of the risk of not asking you out. Nietzsche and deadlifts. It makes sense if you know me."},
    {"person": "Veterinarian, 28, Denver, hikes fourteeners, plays piano", "bio": "I'll handle your injured raccoon and then go summit a 14er. Piano is how I recover from both. Looking for someone who's comfortable with all three."},
    {"person": "Product manager, 34, NYC, salsa dancer, speaks four languages", "bio": "I run roadmaps during the week and salsa floors on weekends. Four languages and still struggling to say 'I like you' at the right moment."},
    {"person": "Marine biologist, 26, Miami, spearfishes, plays drums", "bio": "I track fish for science and catch them for dinner. The drums are how I process the fact that I've seen a whale shark up close and you haven't yet."},
    {"person": "Civil engineer, 33, Phoenix, restores motorcycles, loves spicy food", "bio": "I build bridges professionally and try not to burn them personally. Currently: restoring a '74 Honda and searching for the hottest wings in Phoenix."},
    {"person": "Librarian, 29, Minneapolis, writes poetry, into vintage fashion", "bio": "I shelve books by day and write bad poems at midnight. The vintage coat is mine, the library card is yours if you want it."},
    {"person": "Data scientist, 27, SF, rock climbs, brews kombucha", "bio": "I find patterns in data and chaos on a climbing wall. My kombucha is currently on its third culture. I am not the same person I was a year ago."},
    {"person": "Architect, 35, Chicago, sketches cities, sails on Lake Michigan", "bio": "I design buildings that outlast me and sail to remember that I can't control everything. Bring good coffee and strong opinions."},
    {"person": "Journalist, 28, DC, speaks Arabic, does judo", "bio": "I ask hard questions for a living and throw people for sport. Fluent in Arabic, sarcasm, and deadline panic."},
    {"person": "Electrician, 30, Detroit, raises chickens, is into sci-fi novels", "bio": "I wire houses by day and collect eggs at 6am. Currently reading Dune for the fourth time. The chickens have opinions."},
    {"person": "Physical therapist, 31, Nashville, country line dances, bakes pies", "bio": "I fix knees professionally and two-step recreationally. Pie-baking is how I make friends. It works better than you'd expect."},
    {"person": "Startup founder, 29, Austin, does Brazilian jiu-jitsu, fixes old cameras", "bio": "I'm building something that might fail. The jiu-jitsu prepares me for that. The cameras remind me what's worth preserving."}
]

data_dir = Path("/content/data")
data_dir.mkdir(exist_ok=True)

sft_path = data_dir / "sft_train.jsonl"
with open(sft_path, "w") as f:
    for record in SYNTHETIC_BIOS:
        f.write(json.dumps(record) + "\n")

print(f"Wrote {len(SYNTHETIC_BIOS)} examples to {sft_path}")

In [ ]:
# Load model + tokenizer, configure LoRA
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

MODEL_ID = "distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
)

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=32,
    target_modules=["c_attn"],
    lora_dropout=0.05,
    bias="none",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# Load dataset and train with SFTTrainer
import sys
sys.path.insert(0, str(repo_root))

from src.dataset import load_sft_dataset
from trl import SFTTrainer, SFTConfig

dataset = load_sft_dataset(sft_path)
print(f"Train: {len(dataset['train'])} examples, Val: {len(dataset['val'])} examples")

sft_config = SFTConfig(
    output_dir="/content/sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=8,
    learning_rate=2e-4,
    max_length=256,
    logging_steps=5,
    save_strategy="epoch",
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
)

trainer.train()

In [ ]:
# Save model and publish metrics
import json
from pathlib import Path

# Save the fine-tuned model
sft_checkpoint = repo_root / "results" / "sft_checkpoint"
trainer.save_model(str(sft_checkpoint))
tokenizer.save_pretrained(str(sft_checkpoint))
print(f"Model saved to {sft_checkpoint}")

# Extract and save training metrics
results_dir = paths["results"]
results_dir.mkdir(parents=True, exist_ok=True)

log_history = trainer.state.log_history
train_losses = [e for e in log_history if "loss" in e]

metrics = {
    "model": "distilgpt2",
    "lora_r": 8,
    "epochs": 3,
    "batch_size": 8,
    "learning_rate": 2e-4,
    "num_train_examples": len(dataset["train"]),
    "final_loss": train_losses[-1].get("loss") if train_losses else None,
    "log_history": train_losses,
}

metrics_path = results_dir / "sft_metrics.json"
metrics_path.write_text(json.dumps(metrics, indent=2))
print(f"Metrics saved to {metrics_path}")

publish_artifacts(
    [metrics_path],
    "add SFT training metrics",
    repo_root,
)
print("Done.")